This notebook contains utility procedures used during spoof dataset preparation.
The workflow includes:

1. Adding mapped age labels to spoof-target manifests.
2. Merging manifests and metadata files when required.
3. Extracting transcripts for spoof-source segments.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# GitHub username and repository name
USERNAME = "ArwaFadaaq"
REPO = "Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat"

# Clone public repository
!git clone https://github.com/{USERNAME}/{REPO}.git

# Enter repository folder
%cd Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat

Cloning into 'Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat'...
remote: Enumerating objects: 1520, done.
remote: Counting objects: 100% (438/438), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 1520 (delta 393), reused 213 (delta 213), pack-reused 1082 (from 2)
Receiving objects: 100% (1520/1520), 2.82 MiB | 12.27 MiB/s, done.
Resolving deltas: 100% (909/909), done.
/content/Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat


In [8]:
import pandas as pd


This function adds a mapped age class column to the spoof-target manifest by extracting the age label (adult/minor) from the 'pool' column. This is used to facilitate later dataset construction for spoof generation


In [4]:

def add_mapped_age_class(manifest_path, pool_column="pool"):
    """
    Adds a column 'mapped_age_class' extracted from the pool column
    and saves the dataframe back to the same file path.
    """
    df = pd.read_csv(manifest_path)

    df["mapped_age_class"] = df[pool_column].astype(str).str.split("_").str[0]

    df.to_csv(manifest_path, index=False)

    print(f"Updated file saved to: {manifest_path}")

    return df

In [7]:
train = add_mapped_age_class("/content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/train_spoof_targets.csv")
valid = add_mapped_age_class("/content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/val_spoof_targets.csv")
test = add_mapped_age_class("/content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/test_spoof_targets.csv")

Updated file saved to: /content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/train_spoof_targets.csv
Updated file saved to: /content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/val_spoof_targets.csv
Updated file saved to: /content/drive/MyDrive/age verification/processed/manifest/spoof_targets_splits/test_spoof_targets.csv


# **Transcript Extraction**

In [10]:
# ============================================================
# Merge Manifest Files for Transcript Extraction
# ============================================================
# Multiple manifest files are merged into unified manifests to simplify
# transcript extraction and subsequent processing.

import pandas as pd


def merge_manifests(manifest_a, manifest_b, output_path):
    df_a = pd.read_csv(manifest_a)
    df_b = pd.read_csv(manifest_b)

    merged_df = pd.concat([df_a, df_b], ignore_index=True)
    merged_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print(f"Rows: {len(merged_df):,}\n")


# ------------------------------------------------------------
# Merge file manifests
# ------------------------------------------------------------

merge_manifests(
    manifest_a="/content/drive/MyDrive/age verification/processed/manifest/file_manifest.csv",
    manifest_b="/content/drive/MyDrive/age verification/processed/manifest/cv_backup_file_manifest.csv",
    output_path="/content/drive/MyDrive/age verification/spoofing/utils/file_manifest_with_cvbakcup_merged.csv"
)


# ------------------------------------------------------------
# Merge transcript metadata manifests
# ------------------------------------------------------------

merge_manifests(
    manifest_a="/content/drive/MyDrive/age verification/data/common_voice/data_extraction/cv_extracted_final_metadata.csv",
    manifest_b="/content/drive/MyDrive/age verification/data/common_voice/data_extraction_cv_backup/cv_backup_extracted_metadata.csv",
    output_path="/content/drive/MyDrive/age verification/spoofing/utils/cv_extracted_final_metadata_merged_with_backup.csv"
)

/tmp/ipykernel_4080/3540850135.py:11: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_a = pd.read_csv(manifest_a)


Saved: /content/drive/MyDrive/age verification/spoofing/utils/file_manifest_with_cvbakcup_merged.csv
Rows: 370,036



/tmp/ipykernel_4080/3540850135.py:11: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_a = pd.read_csv(manifest_a)
/tmp/ipykernel_4080/3540850135.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_b = pd.read_csv(manifest_b)


Saved: /content/drive/MyDrive/age verification/spoofing/utils/cv_extracted_final_metadata_merged_with_backup.csv
Rows: 200,681



In [ ]:
from Spoofing.build_transcript import build_transcript_inventory


In [ ]:
result = build_transcript_inventory(
    source_csv="/content/drive/MyDrive/age verification/processed/manifest/real_clean_splits/final_split/train_spoof_source_clean.csv",

    manifest_csv="/content/drive/MyDrive/age verification/spoofing/utils/file_manifest_with_cvbakcup_merged.csv",

    metadata_csv="/content/drive/MyDrive/age verification/spoofing/utils/cv_extracted_final_metadata_merged_with_backup.csv",

    myst_root_dir="/content/drive/MyDrive/age verification/data/myst/data",

    output_csv="/content/drive/MyDrive/age verification/spoofing/transcripts/train_spoof_c_transcript_inventory.csv"
)

/content/Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat/Spoofing/build_transcript.py:130: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta     = pd.read_csv(metadata_csv)


CV rows:   8690
MYST rows: 4439

🔄 Processing transcripts...

✅ Saved transcript inventory:
   Path:             /content/drive/MyDrive/age verification/spoofing/transcripts/train_spoof_c_transcript_inventory.csv
   Total rows:       13129
   With transcript:  10943
   Without:          2186

🧹 Cleaning stats:
   Had raw text:      10945
   Survived cleaning: 10943
   Lost to cleaning:  2


In [ ]:
result = build_transcript_inventory(
    source_csv="/content/drive/MyDrive/age verification/processed/manifest/real_clean_splits/final_split/valid_spoof_source_clean.csv",

    manifest_csv="/content/drive/MyDrive/age verification/spoofing/utils/file_manifest_with_cvbakcup_merged.csv",

    metadata_csv="/content/drive/MyDrive/age verification/spoofing/utils/cv_extracted_final_metadata_merged_with_backup.csv",

    myst_root_dir="/content/drive/MyDrive/age verification/data/myst/data",

    output_csv="/content/drive/MyDrive/age verification/spoofing/transcripts/valid_spoof_c_transcript_inventory.csv"
)

/content/Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat/Spoofing/build_transcript.py:130: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta     = pd.read_csv(metadata_csv)


CV rows:   1445
MYST rows: 922

🔄 Processing transcripts...

✅ Saved transcript inventory:
   Path:             /content/drive/MyDrive/age verification/spoofing/transcripts/valid_spoof_c_transcript_inventory.csv
   Total rows:       2367
   With transcript:  1850
   Without:          517

🧹 Cleaning stats:
   Had raw text:      1850
   Survived cleaning: 1850
   Lost to cleaning:  0


In [ ]:
result = build_transcript_inventory(
    source_csv="/content/drive/MyDrive/age verification/processed/manifest/real_clean_splits/final_split/test_spoof_source_clean.csv",

    manifest_csv="/content/drive/MyDrive/age verification/spoofing/utils/file_manifest_with_cvbakcup_merged.csv",

    metadata_csv="/content/drive/MyDrive/age verification/spoofing/utils/cv_extracted_final_metadata_merged_with_backup.csv",

    myst_root_dir="/content/drive/MyDrive/age verification/data/myst/data",

    output_csv="/content/drive/MyDrive/age verification/spoofing/transcripts/test_spoof_c_transcript_inventory.csv"
)

/content/Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat/Spoofing/build_transcript.py:130: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta     = pd.read_csv(metadata_csv)


CV rows:   1426
MYST rows: 910

🔄 Processing transcripts...

✅ Saved transcript inventory:
   Path:             /content/drive/MyDrive/age verification/spoofing/transcripts/test_spoof_c_transcript_inventory.csv
   Total rows:       2336
   With transcript:  1903
   Without:          433

🧹 Cleaning stats:
   Had raw text:      1903
   Survived cleaning: 1903
   Lost to cleaning:  0
